# 04 - Inference Demo (Nepal tile)

End-to-end prediction on a real Nepal tile (`nepal_s2_253.h5` - the Lamosangu-Jiri Road sector in Sindhupalchok, kept out of training as part of the val split):

1. build the 42-channel standardized input
2. run the U-Net with test-time augmentation
3. apply the calibrated threshold and the NDVI gate
4. produce the insight report (areas, blob stats, road-blockage check)

> This is exactly the pipeline behind the Streamlit dashboard.

In [1]:
import os, sys, warnings
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from paths import PATHS
from predict import execute_vision_inference_pass, resolve_image_path, get_post_rgb

In [2]:
if not os.path.exists(PATHS.WEIGHTS):
    print("ERROR: model weights not found. Run `python tools/download_weights.py` or train first.")
else:
    print("Weights OK:", PATHS.WEIGHTS)

Weights OK: D:\COSMOEYE\models\landslide_unet_weights.pth


In [3]:
SAMPLE = "nepal_s2_253.h5"
print("Tile location:", resolve_image_path(SAMPLE))

Tile location: D:\COSMOEYE\data\TestData\img\nepal_s2_253.h5


In [4]:
spatial_metrics, binary_mask, insight = execute_vision_inference_pass(SAMPLE, save_png=True, verbose=True)

================== GEOSPATIAL ANALYSIS PASS COMPLETE ==================
Processed Target File: nepal_s2_253.h5
Valid POST observations: 100.0%
  ROAD BLOCKAGE: Lamosangu-Jiri Road within 22 m
Discovered Hazard Anomalies Records: [{'object_id': 1, 'centroid_pixel': (61, 89), 'surface_area_sqm': 400}, {'object_id': 2, 'centroid_pixel': (55, 81), 'surface_area_sqm': 650}, {'object_id': 4, 'centroid_pixel': (45, 64), 'surface_area_sqm': 3400}]

==================== ENGLISH REPORT ====================
[TRIAGE AID ONLY] IMPORTANT: This output is a machine-learning triage aid intended to help prioritise field surveys — it is NOT a confirmed hazard assessment. All detections must be verified by a qualified geohazard specialist before any emergency response or infrastructure decisions are made.

LANDSLIDE INSIGHT — nepal_s2_253.h5
Verdict: LANDSLIDE DETECTED (4 separate region(s))
  Landslide pixels: 70 (0.43 % of scene)
  Estimated area: 7,000 sq m (0.70 ha)
  Largest body: 4,600 sq m
  Detect

In [5]:
print(insight)

{'detected_pixels': 70, 'scene_pixels': 16384, 'scene_coverage_pct': 0.42724609375, 'area_sqm': 7000.0, 'area_ha': 0.7, 'n_blobs': 4, 'largest_blob_area_sqm': 4600.0, 'regions': [{'area_sqm': 4600.0, 'prob_mean_pct': 40.84922969341278, 'prob_max_pct': 63.10932636260986}, {'area_sqm': 1200.0, 'prob_mean_pct': 35.56694686412811, 'prob_max_pct': 51.64368152618408}, {'area_sqm': 900.0, 'prob_mean_pct': 31.098896265029907, 'prob_max_pct': 41.325438022613525}, {'area_sqm': 300.0, 'prob_mean_pct': 25.883403420448303, 'prob_max_pct': 28.95653247833252}], 'conf_mean': 0.3804869055747986, 'conf_max': 0.6310932636260986, 'ndvi_mean_det': 0.3428015112876892, 'ndvi_min_scene': -0.041119858622550964, 'ndvi_mean_scene': 0.5345360040664673, 'prob_img': array([[0.0306624 , 0.03211592, 0.03350391, ..., 0.02179205, 0.02252526,
        0.02238403],
       [0.03188537, 0.03336363, 0.03424387, ..., 0.02212101, 0.02299692,
        0.02296655],
       [0.02991128, 0.03207232, 0.03248589, ..., 0.0226618 , 0.02

## Visualization

POST RGB with the predicted landslide mask outlined, plus the raw probability field.

In [6]:
from predict import get_cached_model, tta_probabilities, build_standardized_input, load_normalization_stats
from predict import normalize_standardized_image
from dataset import stack_temporal_stacks
import h5py, torch

model = get_cached_model()
with h5py.File(resolve_image_path(SAMPLE), "r") as f:
    raw = np.array(f["img"])
mean, std = load_normalization_stats()
fused = build_standardized_input(raw)
normalized = normalize_standardized_image(fused, mean, std)
tensor = torch.from_numpy(stack_temporal_stacks(normalized)).unsqueeze(0).to(next(model.parameters()).device)
probs = tta_probabilities(model, tensor, next(model.parameters()).device).squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(get_post_rgb(SAMPLE)); axes[0].set_title("POST RGB")
axes[1].imshow(probs, cmap="hot", vmin=0, vmax=1); axes[1].set_title("Model probability")
axes[2].imshow(get_post_rgb(SAMPLE))
from predict import mask_overlay_rgba
axes[2].imshow(mask_overlay_rgba(binary_mask, alpha=0.55)); axes[2].set_title(f"Predicted landslides ({binary_mask.sum()} px)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [307.0..2454.0].


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [307.0..2454.0].


## Result

The panel PNGs written by the dashboard pipeline live in `results/insights/`; the insight dict above shows detected area, largest blob, scene coverage, and whether any detections sit on a mapped road (checked only for Nepal tiles).